In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-13 22:11:29--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.226.36.218, 13.226.36.73, 13.226.36.196, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.226.36.218|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  37.7MB/s    in 1.6s    

2025-03-13 22:11:31 (37.7 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()

25/03/13 22:28:14 WARN Utils: Your hostname, main resolves to a loopback address: 127.0.1.1; using 192.168.1.95 instead (on interface enp1s0)
25/03/13 22:28:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/13 22:28:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
pwd

'/home/adam2eden'

In [12]:
df = spark.read.parquet("/home/adam2eden/Workspace/de2025/week5/yellow_tripdata_2024-10.parquet")

In [18]:
df = df.repartition(4)

In [19]:
df.write.parquet("tripdata/2024/10")

In [25]:
ls -lh tripdata/2024/10/

total 90M
-rw-r--r-- 1 adam2eden adam2eden 23M Mar 13 22:35 part-00000-43910514-79b5-4358-a0ea-ae5173f43364-c000.snappy.parquet
-rw-r--r-- 1 adam2eden adam2eden 23M Mar 13 22:35 part-00001-43910514-79b5-4358-a0ea-ae5173f43364-c000.snappy.parquet
-rw-r--r-- 1 adam2eden adam2eden 23M Mar 13 22:35 part-00002-43910514-79b5-4358-a0ea-ae5173f43364-c000.snappy.parquet
-rw-r--r-- 1 adam2eden adam2eden 23M Mar 13 22:35 part-00003-43910514-79b5-4358-a0ea-ae5173f43364-c000.snappy.parquet
-rw-r--r-- 1 adam2eden adam2eden   0 Mar 13 22:35 _SUCCESS


In [37]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [30]:
df.registerTempTable("tripdata")

/home/adam2eden/env/datatalks/lib/python3.12/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [38]:
spark.sql("""
SELECT COUNT(1)
FROM tripdata
WHERE date_format(tpep_pickup_datetime, 'd') = 15
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [40]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [47]:
spark.sql("""
SELECT 
tpep_pickup_datetime, tpep_dropoff_datetime
FROM tripdata
LIMIT 10
""").show()

+--------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|
+--------------------+---------------------+
| 2024-10-03 14:00:53|  2024-10-03 14:11:51|
| 2024-10-08 08:14:12|  2024-10-08 08:26:56|
| 2024-10-09 06:56:06|  2024-10-09 06:59:32|
| 2024-10-08 12:36:28|  2024-10-08 12:44:42|
| 2024-10-08 21:56:55|  2024-10-08 22:13:14|
| 2024-10-04 20:42:46|  2024-10-04 21:00:54|
| 2024-10-04 07:57:45|  2024-10-04 08:04:27|
| 2024-10-04 15:55:55|  2024-10-04 15:56:01|
| 2024-10-06 00:58:36|  2024-10-06 01:06:37|
| 2024-10-08 18:25:34|  2024-10-08 18:30:47|
+--------------------+---------------------+



In [ ]:
spark.sql("""
SELECT 
max(round((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600, 2))
FROM tripdata
""").show()

+------------------------------------------------------------------------------------------------------------------------------------------------+
|max(round(((unix_timestamp(tpep_dropoff_datetime, yyyy-MM-dd HH:mm:ss) - unix_timestamp(tpep_pickup_datetime, yyyy-MM-dd HH:mm:ss)) / 3600), 2))|
+------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                                          162.62|
+------------------------------------------------------------------------------------------------------------------------------------------------+



In [53]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-13 22:57:35--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.226.36.196, 13.226.36.218, 13.226.36.130, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.226.36.196|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2025-03-13 22:57:35 (11.9 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [56]:
zone = spark.read.option("header","true").csv("taxi_zone_lookup.csv")

In [57]:
zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [58]:
zone.registerTempTable("zone")

/home/adam2eden/env/datatalks/lib/python3.12/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [64]:
spark.sql("""
WITH CTE AS (
    SELECT 
    COUNT(1) AS count, PULocationID
    FROM tripdata
    GROUP BY PULocationID
    ORDER by count asc 
    LIMIT 1
)
SELECT z.Zone
FROM zone z
JOIN CTE on z.LocationID = CTE.PULocationID
""").show()

+--------------------+
|                Zone|
+--------------------+
|Governor's Island...|
+--------------------+

